In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install catboost --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 26.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, log_loss, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
import warnings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

warnings.filterwarnings('ignore')
SEED = 33

In [ ]:
from enum import auto
from catboost import CatBoostClassifier,CatBoostRegressor, Pool


def load_data(path='data/'):
    train_df = pd.read_csv(path + 'Train.csv')
    test_df = pd.read_csv(path + 'Test.csv')
    sku_df = pd.read_csv(path + 'sku_data.csv')
    customer_df = pd.read_csv(path + 'customer_data.csv')

    return train_df, test_df, sku_df, customer_df

def preprocess_dates(df):
    date_cols = ['week_start', 'customer_created_at']
    for col in date_cols:
        df[col] = pd.to_datetime(df[col])

    return df

def robust_feature_engineering_with_lags(train_df, test_df, sku_df, customer_df):
    # Same merges as before
    train_df = pd.merge(train_df, sku_df, on=['product_unit_variant_id', 'unit_name', 'grade_name'], how='left')
    test_df = pd.merge(test_df, sku_df, on=['product_unit_variant_id', 'unit_name', 'grade_name'], how='left')

    train_df = pd.merge(train_df, customer_df, on='customer_id', how='left', suffixes=('', '_y'))
    test_df = pd.merge(test_df, customer_df, on='customer_id', how='left', suffixes=('', '_y'))

    for df in [train_df, test_df]:
        drop_cols = [col for col in df.columns if '_y' in col]
        if drop_cols: df.drop(columns=drop_cols, inplace=True)
        df = preprocess_dates(df)

    # Concat for Lag Generation
    train_df['is_train'] = 1
    test_df['is_train'] = 0

    # Common columns for concat
    full_df = pd.concat([train_df, test_df], ignore_index=True)

    full_df = full_df.sort_values(by=['customer_id', 'product_unit_variant_id', 'week_start'])

    lag_features = ['qty_this_week', 'num_orders_week', 'spend_this_week', 'purchased_this_week', 'Target_purchase_next_2w', 'Target_qty_next_2w']
    lag_periods = [2, 3, 4, 5, 6, 7, 8]

    print("Generating lags on full history...")
    for feature in lag_features:
        for lag in lag_periods:
            full_df[f'{feature}_lag_{lag}w'] = full_df.groupby(['customer_id', 'product_unit_variant_id'])[feature].shift(lag)

    print("Generating week-over-week features...")

    grouped = full_df.groupby(['customer_id', 'product_unit_variant_id'])

    full_df['weeks_since_last_purchase'] = grouped['purchased_this_week'] \
        .transform(lambda x: x.shift(1).apply(lambda v: 0 if v == 1 else np.nan).cumsum())

    full_df['last_purchase_qty'] = grouped['qty_this_week'] \
        .transform(lambda x: x.shift(1).where(x.shift(1) > 0).ffill())

    full_df['last_purchase_spend'] = grouped['spend_this_week'] \
        .transform(lambda x: x.shift(1).where(x.shift(1) > 0).ffill())


    print("Generating rolling features on full history...")
    rolling_windows = [4, 8]
    for feature in ['qty_this_week', 'num_orders_week', 'spend_this_week','Target_purchase_next_1w', 'Target_qty_next_1w','Target_purchase_next_2w', 'Target_qty_next_2w']:
        grouped = full_df.groupby(['customer_id', 'product_unit_variant_id'])[feature]
        for window in rolling_windows:
          if feature not in ['Target_purchase_next_2w', 'Target_qty_next_2w']:
            full_df[f'{feature}_rolling_mean_{window}w'] = grouped.transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
            full_df[f'{feature}_rolling_std_{window}w'] = grouped.transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).std())
            full_df[f'{feature}_rolling_min_{window}w'] = grouped.transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).min())
            full_df[f'{feature}_rolling_max_{window}w'] = grouped.transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).max())

          else:
            full_df[f'{feature}_rolling_mean_{window}w_shift2'] = grouped.transform(lambda x: x.shift(2).rolling(window=window, min_periods=1).mean())
            full_df[f'{feature}_rolling_std_{window}w_shift2'] = grouped.transform(lambda x: x.shift(2).rolling(window=window, min_periods=1).std())
            full_df[f'{feature}_rolling_min_{window}w_shift2'] = grouped.transform(lambda x: x.shift(2).rolling(window=window, min_periods=1).min())
            full_df[f'{feature}_rolling_max_{window}w_shift2'] = grouped.transform(lambda x: x.shift(2).rolling(window=window, min_periods=1).max())

        #expanding features

        if feature not in ['Target_purchase_next_2w', 'Target_qty_next_2w']:
            full_df[f'{feature}_expanding_mean_1w'] = grouped.transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
            full_df[f'{feature}_expanding_std_1w'] = grouped.transform(lambda x: x.shift(1).expanding(min_periods=1).std())
            full_df[f'{feature}_expanding_min_1w'] = grouped.transform(lambda x: x.shift(1).expanding(min_periods=1).min())
            full_df[f'{feature}_expanding_max_1w'] = grouped.transform(lambda x: x.shift(1).expanding(min_periods=1).max())

        else:
            full_df[f'{feature}_expanding_mean_2w'] = grouped.transform(lambda x: x.shift(2).expanding(min_periods=1).mean())
            full_df[f'{feature}_expanding_std_2w'] = grouped.transform(lambda x: x.shift(2).expanding(min_periods=1).std())
            full_df[f'{feature}_expanding_min_2w'] = grouped.transform(lambda x: x.shift(2).expanding(min_periods=1).min())
            full_df[f'{feature}_expanding_max_2w'] = grouped.transform(lambda x: x.shift(2).expanding(min_periods=1).max())

    for w in [4, 8]:
        full_df[f'qty_trend_{w}w'] = (
            full_df[f'qty_this_week_rolling_mean_{w}w'] -
            full_df[f'qty_this_week_rolling_mean_{w}w'].shift(w)
        )

        full_df[f'spend_trend_{w}w'] = (
            full_df[f'spend_this_week_rolling_mean_{w}w'] -
            full_df[f'spend_this_week_rolling_mean_{w}w'].shift(w)
        )

    # Date parts
    full_df['week_start_year'] = full_df['week_start'].dt.year
    full_df['week_start_month'] = full_df['week_start'].dt.month
    full_df['week_start_weekofyear'] = full_df['week_start'].dt.isocalendar().week.astype(int)
    full_df['week_start_quarter'] = full_df['week_start'].dt.quarter

    full_df['customer_tenure_days'] = (full_df['week_start'] - full_df['customer_created_at']).dt.days

    train_subset = full_df[full_df['is_train'] == 1]
    agg_features = ['qty_this_week', 'num_orders_week', 'spend_this_week', 'selling_price']
    agg_funcs = ['mean', 'sum']

    # By customer
    for func in agg_funcs:
        agg = full_df.groupby('customer_id')[agg_features].agg(func).reset_index()
        agg.columns = ['customer_id'] + [f'customer_{col}_{func}' for col in agg_features]
        full_df = pd.merge(full_df, agg, on='customer_id', how='left')

    # By product
    for func in agg_funcs:
        agg = full_df.groupby('product_id')[agg_features].agg(func).reset_index()
        agg.columns = ['product_id'] + [f'product_{col}_{func}' for col in agg_features]
        full_df = pd.merge(full_df, agg, on='product_id', how='left')

    # By product_unit_variant
    for func in agg_funcs:
        agg = full_df.groupby('product_unit_variant_id')[agg_features].agg(func).reset_index()
        agg.columns = ['product_unit_variant_id'] + [f'product_unit_variant_{col}_{func}' for col in agg_features]
        full_df = pd.merge(full_df, agg, on='product_unit_variant_id', how='left')

    full_df['qty_vs_customer_avg'] = (
        full_df['qty_this_week_lag_2w'] /
        (full_df['customer_qty_this_week_mean'] + 1e-6)
    )

    full_df['spend_vs_customer_avg'] = (
        full_df['spend_this_week_lag_2w'] /
        (full_df['customer_spend_this_week_mean'] + 1e-6)
    )


    full_df['qty_vs_product_avg'] = (
        full_df['qty_this_week_lag_2w'] /
        (full_df['product_qty_this_week_mean'] + 1e-6)
    )
    full_df['is_new_customer'] = (full_df['customer_tenure_days'] < 30).astype(int)

    full_df['history_length_weeks'] = (
        full_df.groupby(['customer_id', 'product_unit_variant_id']).cumcount()
    )

    full_df['low_history_flag'] = (full_df['history_length_weeks'] < 4).astype(int)

    # Fill NaNs
    fill_0_cols = [c for c in full_df.columns if '_lag_' in c or '_rolling_' in c or '_mean' in c or '_sum' in c]
    full_df[fill_0_cols] = full_df[fill_0_cols].fillna(0)

    cat_cols = ['product_name', 'product_grade_variant_sku']
    for col in cat_cols:
        full_df[col] = full_df[col].fillna('Unknown')
    full_df['grade_active_status'] = full_df['grade_active_status'].fillna(False).astype(bool)

    # Split back
    train_df_processed = full_df[full_df['is_train'] == 1].copy()
    test_df_processed = full_df[full_df['is_train'] == 0].copy()

    return train_df_processed, test_df_processed

def train_and_predict_horizon_stacked(train_df, test_df, seed=42):
    print("Initializing Gap-Aware Stacked Training...")

    # 1. Column Definitions
    lag_1_cols = [c for c in train_df.columns if '_lag_2' in c]
    lag_2_cols = [c for c in train_df.columns if '_lag_3' in c]

    cols_h1 = {'Target_purchase_next_1w': 'target_purchase', 'Target_qty_next_1w': 'target_qty'}
    cols_h2 = {'Target_purchase_next_2w': 'target_purchase', 'Target_qty_next_2w': 'target_qty'}
    drop_targets = list(cols_h1.keys()) + list(cols_h2.keys())

    ignore_cols = ['ID', 'is_train', 'week_start', 'customer_created_at', 'target_purchase', 'target_qty'] + drop_targets
    current_obs = ['qty_this_week', 'num_orders_week', 'spend_this_week', 'purchased_this_week']

    # Features that exist in the raw dataframe
    base_features = [c for c in train_df.columns if c not in (ignore_cols + current_obs)]
    cat_feats = train_df[base_features].select_dtypes(include=['object', 'bool', 'category']).columns.tolist()

    # Final feature list for the model (includes 'horizon' created during stacking)
    model_features = base_features + ['horizon']

    regression_features = ['customer_id', 'product_unit_variant_id', 'product_id', 'grade_name', 'unit_name', 'product_grade_variant_id', 'selling_price', 'customer_category', 'customer_status',
                           'product_name', 'product_grade_variant_sku', 'grade_active_status', 'qty_this_week_lag_2w', 'qty_this_week_lag_3w', 'qty_this_week_lag_4w', 'qty_this_week_lag_5w',
                           'qty_this_week_lag_6w', 'qty_this_week_lag_7w', 'qty_this_week_lag_8w', 'num_orders_week_lag_2w', 'num_orders_week_lag_3w', 'num_orders_week_lag_4w',
                           'num_orders_week_lag_5w', 'num_orders_week_lag_6w', 'num_orders_week_lag_7w', 'num_orders_week_lag_8w', 'spend_this_week_lag_2w', 'spend_this_week_lag_3w',
                           'spend_this_week_lag_4w', 'spend_this_week_lag_5w', 'spend_this_week_lag_6w', 'spend_this_week_lag_7w', 'spend_this_week_lag_8w', 'purchased_this_week_lag_2w',
                           'purchased_this_week_lag_3w', 'purchased_this_week_lag_4w', 'purchased_this_week_lag_5w', 'purchased_this_week_lag_6w', 'purchased_this_week_lag_7w',
                           'purchased_this_week_lag_8w', 'Target_purchase_next_2w_lag_2w', 'Target_purchase_next_2w_lag_3w', 'Target_purchase_next_2w_lag_4w', 'Target_purchase_next_2w_lag_5w',
                           'Target_purchase_next_2w_lag_6w', 'Target_purchase_next_2w_lag_7w', 'Target_purchase_next_2w_lag_8w', 'Target_qty_next_2w_lag_2w', 'Target_qty_next_2w_lag_3w',
                           'Target_qty_next_2w_lag_4w', 'Target_qty_next_2w_lag_5w', 'Target_qty_next_2w_lag_6w', 'Target_qty_next_2w_lag_7w', 'Target_qty_next_2w_lag_8w',
                           'qty_this_week_rolling_mean_4w', 'qty_this_week_rolling_std_4w', 'qty_this_week_rolling_mean_8w', 'qty_this_week_rolling_std_8w', 'num_orders_week_rolling_mean_4w',
                           'num_orders_week_rolling_std_4w', 'num_orders_week_rolling_mean_8w', 'num_orders_week_rolling_std_8w', 'spend_this_week_rolling_mean_4w', 'spend_this_week_rolling_std_4w',
                           'spend_this_week_rolling_mean_8w', 'spend_this_week_rolling_std_8w', 'week_start_year', 'week_start_month', 'week_start_weekofyear', 'week_start_quarter',
                           'customer_tenure_days', 'customer_qty_this_week_mean', 'customer_num_orders_week_mean', 'customer_spend_this_week_mean', 'customer_selling_price_mean',
                           'customer_qty_this_week_sum', 'customer_num_orders_week_sum', 'customer_spend_this_week_sum', 'customer_selling_price_sum', 'product_qty_this_week_mean',
                           'product_num_orders_week_mean', 'product_spend_this_week_mean', 'product_selling_price_mean', 'product_qty_this_week_sum', 'product_num_orders_week_sum',
                           'product_spend_this_week_sum', 'product_selling_price_sum', 'product_unit_variant_qty_this_week_mean', 'product_unit_variant_num_orders_week_mean',
                           'product_unit_variant_spend_this_week_mean', 'product_unit_variant_selling_price_mean', 'product_unit_variant_qty_this_week_sum', 'product_unit_variant_num_orders_week_sum',
                           'product_unit_variant_spend_this_week_sum', 'product_unit_variant_selling_price_sum', 'horizon']
    # 2. Parameters
    clf_params = {
        'objective': 'binary', 'metric': 'binary_logloss', 'n_estimators': 2000,
        'learning_rate': 0.03, 'num_leaves': 31, 'feature_fraction': 0.8,
        'bagging_fraction': 0.8, 'bagging_freq': 5, 'n_jobs': -1,
        'random_state': seed, 'verbose': -1
    }
    reg_params = {
        'objective': 'regression_l1', 'metric': 'l1', 'n_estimators': 2000,
        'learning_rate': 0.08, 'num_leaves': 31, 'feature_fraction': 0.8,
        'bagging_fraction': 0.8, 'bagging_freq': 5, 'n_jobs': -1,
        'random_state': seed, 'verbose': -1
    }

    clf_params =  {'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt', 'verbosity': -1, 'n_jobs': -1, 'random_state': 42,
                   'n_estimators': 247, 'learning_rate': 0.01035703290344576, 'num_leaves': 55, 'max_depth': 0, 'min_child_samples': 75,
                   'subsample': 0.7116436610081205, 'subsample_freq': 7, 'colsample_bytree': 0.9458969750455872, 'reg_alpha': 0.005137395475753478,
                   'reg_lambda': 5.116135509305144e-06, 'min_split_gain': 0.5328034350030296, 'is_unbalance': False, 'scale_pos_weight': 3.1055339600014347}


    cat_clf_params = dict(
        iterations=2000,              # upper bound
        learning_rate=0.03,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        task_type="GPU",
        devices="0",
        auto_class_weights = 'SqrtBalanced'
    )

    cat_reg_params = dict(
        iterations=2000,              # upper bound
        learning_rate=0.05,
        depth=8,
        loss_function="MAE",          # matches your L1 objective
        eval_metric="MAE",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        task_type="GPU",
        devices="0"
    )


    # 3. Helper: Stack and Mask
    def prepare_stacked_data(df, is_test=False):
        # Horizon 1 (T+2): Mask Lag 1 (the gap week)
        h1 = df.copy()
        if not is_test:
            h1.rename(columns=cols_h1, inplace=True)
            h1 = h1.drop(columns=[c for c in drop_targets if c not in cols_h1], errors='ignore')
        h1['horizon'] = 1
        h1[lag_1_cols] = np.nan

        # Horizon 2 (T+3): Mask Lag 1 (H1) & Lag 2 (the gap week)
        h2 = df.copy()
        if not is_test:
            h2.rename(columns=cols_h2, inplace=True)
            h2 = h2.drop(columns=[c for c in drop_targets if c not in cols_h2], errors='ignore')
        h2['horizon'] = 2
        h2[lag_1_cols] = np.nan
        h2[lag_2_cols] = np.nan

        stacked = pd.concat([h1, h2], ignore_index=True)
        for col in cat_feats:
            stacked[col] = stacked[col].astype('category')
        stacked['horizon'] = stacked['horizon'].astype('category')
        return stacked

    # 4. 3-Fold Time-Series Validation
    unique_weeks = sorted(train_df['week_start'].unique())
    cv_metrics = []
    best_iters_clf = []
    best_iters_reg = []
    best_iters_cat_clf = []
    best_iters_cat_reg = []


    print("\nStarting 3-Fold Validation (Mimicking 1-week gap)...")
    for i in range(1, 4):
        # Time pointers: [Train Data] -> [Gap Week] -> [H1 Eval Week] -> [H2 Eval Week]
        idx_h2 = -i
        idx_h1 = -i - 1
        idx_gap = -i - 2
        idx_train_max = -i - 3

        v_h2_df = train_df[train_df['week_start'] == unique_weeks[idx_h2]].copy()
        v_h1_df = train_df[train_df['week_start'] == unique_weeks[idx_h1]].copy()
        t_df = train_df[train_df['week_start'] <= unique_weeks[idx_train_max]].copy()

        # Build Fold Val Set
        v_h1 = v_h1_df.rename(columns=cols_h1).assign(horizon=1)
        v_h1[lag_1_cols] = np.nan
        v_h2 = v_h2_df.rename(columns=cols_h2).assign(horizon=2)
        v_h2[lag_1_cols] = np.nan
        v_h2[lag_2_cols] = np.nan
        val_stacked = pd.concat([v_h1, v_h2], ignore_index=True)

        for col in cat_feats + ['horizon']:
            val_stacked[col] = val_stacked[col].astype('category')

        train_stacked = prepare_stacked_data(t_df)

        # Train Fold Classifier
        clf = lgb.LGBMClassifier(**clf_params)
        clf.fit(
            train_stacked[model_features], train_stacked['target_purchase'],
            eval_set=[(val_stacked[model_features], val_stacked['target_purchase'])],
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )
        best_iters_clf.append(clf.best_iteration_)


        # ---- Train Fold Classifier (CatBoost) ----
        cat_feature_indices = [model_features.index(c) for c in cat_feats + ['horizon'] if c in model_features]

        train_pool = Pool(
            train_stacked[model_features],
            label=train_stacked['target_purchase'],
            cat_features=cat_feature_indices
        )
        val_pool = Pool(
            val_stacked[model_features],
            label=val_stacked['target_purchase'],
            cat_features=cat_feature_indices
        )

        cat_clf = CatBoostClassifier(**cat_clf_params)
        cat_clf.fit(
            train_pool,
            eval_set=val_pool,
            use_best_model=True,
            early_stopping_rounds=50
        )
        best_iters_cat_clf.append(cat_clf.get_best_iteration())


        # Train Fold Regressor
        reg = lgb.LGBMRegressor(**reg_params)
        reg.fit(
            train_stacked[regression_features], train_stacked['target_qty'],
            eval_set=[(val_stacked[regression_features], val_stacked['target_qty'])],
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )
        best_iters_reg.append(reg.best_iteration_)

        # ---- Train Fold Regressor (CatBoost) ----
        reg_cat_feature_indices = [regression_features.index(c) for c in cat_feats + ['horizon'] if c in regression_features]

        train_pool_reg = Pool(
            train_stacked[regression_features],
            label=train_stacked['target_qty'],
            cat_features=reg_cat_feature_indices
        )
        val_pool_reg = Pool(
            val_stacked[regression_features],
            label=val_stacked['target_qty'],
            cat_features=reg_cat_feature_indices
        )

        cat_reg = CatBoostRegressor(**cat_reg_params)
        cat_reg.fit(
            train_pool_reg,
            eval_set=val_pool_reg,
            use_best_model=True,
            early_stopping_rounds=50
        )
        best_iters_cat_reg.append(cat_reg.get_best_iteration())


        # ---- Fold Evaluation (LGB, CAT, ENSEMBLE) ----
        val_stacked['p_prob_lgb'] = clf.predict_proba(val_stacked[model_features])[:, 1]
        val_stacked['p_prob_cat'] = cat_clf.predict_proba(val_stacked[model_features])[:, 1]
        val_stacked['p_prob_ens'] = 0.5 * val_stacked['p_prob_lgb'] + 0.5 * val_stacked['p_prob_cat']
        val_stacked['p_qty_lgb'] = reg.predict(val_stacked[regression_features]).clip(0)
        val_stacked['p_qty_cat'] = cat_reg.predict(val_stacked[regression_features]).clip(0)
        val_stacked['p_qty_ens'] = 0.5 * val_stacked['p_qty_lgb'] + 0.5 * val_stacked['p_qty_cat']


        print(f"\n--- Fold {i} Results ---")
        for h in [1, 2]:
            m = val_stacked['horizon'].astype(int) == h
            h_data = val_stacked[m]

            # Classification metrics
            ll_lgb = log_loss(h_data['target_purchase'], h_data['p_prob_lgb'])
            auc_lgb = roc_auc_score(h_data['target_purchase'], h_data['p_prob_lgb'])

            ll_cat = log_loss(h_data['target_purchase'], h_data['p_prob_cat'])
            auc_cat = roc_auc_score(h_data['target_purchase'], h_data['p_prob_cat'])

            ll_ens = log_loss(h_data['target_purchase'], h_data['p_prob_ens'])
            auc_ens = roc_auc_score(h_data['target_purchase'], h_data['p_prob_ens'])

            # Regression (unchanged)
            # (keep your reg + p_qty logic as-is)
            mae_lgb = mean_absolute_error(h_data['target_qty'], h_data['p_qty_lgb'])
            mae_cat = mean_absolute_error(h_data['target_qty'], h_data['p_qty_cat'])
            mae_ens = mean_absolute_error(h_data['target_qty'], h_data['p_qty_ens'])


            cv_metrics.append({
                'fold': i, 'horizon': h,
                'logloss_lgb': ll_lgb, 'auc_lgb': auc_lgb,
                'logloss_cat': ll_cat, 'auc_cat': auc_cat,
                'logloss_ens': ll_ens, 'auc_ens': auc_ens,
                'mae_lgb': mae_lgb,
                'mae_cat': mae_cat,
                'mae_ens': mae_ens
            })

            print(
                f" Horizon {h} -> "
                f"LGB(LogLoss {ll_lgb:.4f}, AUC {auc_lgb:.4f}) | "
                f"CAT(LogLoss {ll_cat:.4f}, AUC {auc_cat:.4f}) | "
                f"ENS(LogLoss {ll_ens:.4f}, AUC {auc_ens:.4f}) || "
                f"MAE LGB {mae_lgb:.4f} | MAE CAT {mae_cat:.4f} | MAE ENS {mae_ens:.4f}"
            )


    # 5. Display Aggregate CV Results
    cv_res = pd.DataFrame(cv_metrics)
    summary = cv_res.groupby('horizon')[[
        'logloss_lgb','auc_lgb','logloss_cat','auc_cat','logloss_ens','auc_ens',
        'mae_lgb','mae_cat','mae_ens'
    ]].mean()


    print("\n" + "="*55)
    print("AVERAGE PERFORMANCE ACROSS ALL FOLDS")
    print(summary)
    print("="*55)


    # 6. Final Retrain
    avg_iter_clf = int(np.mean(best_iters_clf))
    avg_iter_reg = int(np.mean(best_iters_reg))
    avg_iter_cat_clf = int(np.mean(best_iters_cat_clf))
    avg_iter_cat_reg = int(np.mean(best_iters_cat_reg))


    print(
        f"Best iters -> "
        f"LGB CLF: {avg_iter_clf}, "
        f"LGB REG: {avg_iter_reg}"
        f"CAT CLF: {avg_iter_cat_clf}, "
        f"CAT REG: {avg_iter_cat_reg}"

    )



    full_train_stacked = prepare_stacked_data(train_df)
    final_clf = lgb.LGBMClassifier(**clf_params).set_params(n_estimators=avg_iter_clf)
    final_clf.fit(full_train_stacked[model_features], full_train_stacked['target_purchase'])


    # Final retrain (CatBoost)
    full_train_pool = Pool(
        full_train_stacked[model_features],
        label=full_train_stacked['target_purchase'],
        cat_features=cat_feature_indices
    )
    cat_clf_params['iterations'] = avg_iter_cat_clf
    final_cat_clf = CatBoostClassifier(**cat_clf_params)
    final_cat_clf.fit(full_train_pool)


    final_reg = lgb.LGBMRegressor(**reg_params).set_params(n_estimators=avg_iter_reg)
    final_reg.fit(full_train_stacked[regression_features], full_train_stacked['target_qty'])


    # Final retrain (CatBoost Regressor)
    full_train_pool_reg = Pool(
        full_train_stacked[regression_features],
        label=full_train_stacked['target_qty'],
        cat_features=reg_cat_feature_indices
    )
    cat_reg_params['iterations'] = avg_iter_cat_reg
    final_cat_reg = CatBoostRegressor(**cat_reg_params)
    final_cat_reg.fit(full_train_pool_reg)


    # 7. Test Inference
    print("Generating Test Predictions (T+2 and T+3)...")
    test_stacked = prepare_stacked_data(test_df, is_test=True)
    t_h1 = test_stacked[test_stacked['horizon'].astype(int) == 1]
    t_h2 = test_stacked[test_stacked['horizon'].astype(int) == 2]
    # Purchase probs from both models
    lgb_h1 = final_clf.predict_proba(t_h1[model_features])[:, 1]
    cat_h1 = final_cat_clf.predict_proba(t_h1[model_features])[:, 1]
    ens_h1 = 0.5 * lgb_h1 + 0.5 * cat_h1

    lgb_h2 = final_clf.predict_proba(t_h2[model_features])[:, 1]
    cat_h2 = final_cat_clf.predict_proba(t_h2[model_features])[:, 1]
    ens_h2 = 0.5 * lgb_h2 + 0.5 * cat_h2

    qty_lgb_h1 = final_reg.predict(t_h1[regression_features]).clip(0)
    qty_cat_h1 = final_cat_reg.predict(t_h1[regression_features]).clip(0)
    qty_ens_h1 = 0.5 * qty_lgb_h1 + 0.5 * qty_cat_h1

    qty_lgb_h2 = final_reg.predict(t_h2[regression_features]).clip(0)
    qty_cat_h2 = final_cat_reg.predict(t_h2[regression_features]).clip(0)
    qty_ens_h2 = 0.5 * qty_lgb_h2 + 0.5 * qty_cat_h2

    predictions = {
        # purchase outputs (as you already have)
        'Target_purchase_next_1w_lgb': lgb_h1,
        'Target_purchase_next_1w_cat': cat_h1,
        'Target_purchase_next_1w_ens': ens_h1,

        'Target_purchase_next_2w_lgb': lgb_h2,
        'Target_purchase_next_2w_cat': cat_h2,
        'Target_purchase_next_2w_ens': ens_h2,

        # regression outputs (new)
        'Target_qty_next_1w_lgb': qty_lgb_h1,
        'Target_qty_next_1w_cat': qty_cat_h1,
        'Target_qty_next_1w_ens': qty_ens_h1,

        'Target_qty_next_2w_lgb': qty_lgb_h2,
        'Target_qty_next_2w_cat': qty_cat_h2,
        'Target_qty_next_2w_ens': qty_ens_h2,
    }


    print("Success.")
    return predictions

In [ ]:
print("Loading data...")
train_df, test_df, sku_df, customer_df = load_data('/content/drive/MyDrive/farm_prediction/')

print("Feature engineering...")
train_df, test_df = robust_feature_engineering_with_lags(train_df, test_df, sku_df, customer_df)

display(train_df.head(), test_df.head())

Loading data...
Feature engineering...
Generating lags on full history...
Generating week-over-week features...
Generating rolling features on full history...


,ID,customer_id,product_unit_variant_id,week_start,qty_this_week,num_orders_week,spend_this_week,purchased_this_week,product_id,grade_name,unit_name,product_grade_variant_id,selling_price,customer_category,customer_status,customer_created_at,Target_qty_next_1w,Target_purchase_next_1w,Target_qty_next_2w,Target_purchase_next_2w,product_name,product_grade_variant_sku,grade_active_status,is_train,qty_this_week_lag_2w,qty_this_week_lag_3w,qty_this_week_lag_4w,qty_this_week_lag_5w,qty_this_week_lag_6w,qty_this_week_lag_7w,qty_this_week_lag_8w,num_orders_week_lag_2w,num_orders_week_lag_3w,num_orders_week_lag_4w,num_orders_week_lag_5w,num_orders_week_lag_6w,num_orders_week_lag_7w,num_orders_week_lag_8w,spend_this_week_lag_2w,spend_this_week_lag_3w,spend_this_week_lag_4w,spend_this_week_lag_5w,spend_this_week_lag_6w,spend_this_week_lag_7w,spend_this_week_lag_8w,purchased_this_week_lag_2w,purchased_this_week_lag_3w,purchased_this_week_lag_4w,purchased_this_week_lag_5w,purchased_this_week_lag_6w,purchased_this_week_lag_7w,purchased_this_week_lag_8w,Target_purchase_next_2w_lag_2w,Target_purchase_next_2w_lag_3w,Target_purchase_next_2w_lag_4w,Target_purchase_next_2w_lag_5w,Target_purchase_next_2w_lag_6w,Target_purchase_next_2w_lag_7w,Target_purchase_next_2w_lag_8w,Target_qty_next_2w_lag_2w,Target_qty_next_2w_lag_3w,Target_qty_next_2w_lag_4w,Target_qty_next_2w_lag_5w,Target_qty_next_2w_lag_6w,Target_qty_next_2w_lag_7w,Target_qty_next_2w_lag_8w,weeks_since_last_purchase,last_purchase_qty,last_purchase_spend,qty_this_week_rolling_mean_4w,qty_this_week_rolling_std_4w,qty_this_week_rolling_min_4w,qty_this_week_rolling_max_4w,qty_this_week_rolling_mean_8w,qty_this_week_rolling_std_8w,qty_this_week_rolling_min_8w,qty_this_week_rolling_max_8w,qty_this_week_expanding_mean_1w,qty_this_week_expanding_std_1w,qty_this_week_expanding_min_1w,qty_this_week_expanding_max_1w,num_orders_week_rolling_mean_4w,num_orders_week_rolling_std_4w,num_orders_week_rolling_min_4w,num_orders_week_rolling_max_4w,num_orders_week_rolling_mean_8w,num_orders_week_rolling_std_8w,num_orders_week_rolling_min_8w,num_orders_week_rolling_max_8w,num_orders_week_expanding_mean_1w,num_orders_week_expanding_std_1w,num_orders_week_expanding_min_1w,num_orders_week_expanding_max_1w,spend_this_week_rolling_mean_4w,spend_this_week_rolling_std_4w,spend_this_week_rolling_min_4w,spend_this_week_rolling_max_4w,spend_this_week_rolling_mean_8w,spend_this_week_rolling_std_8w,spend_this_week_rolling_min_8w,spend_this_week_rolling_max_8w,spend_this_week_expanding_mean_1w,spend_this_week_expanding_std_1w,spend_this_week_expanding_min_1w,spend_this_week_expanding_max_1w,Target_purchase_next_1w_rolling_mean_4w,Target_purchase_next_1w_rolling_std_4w,Target_purchase_next_1w_rolling_min_4w,Target_purchase_next_1w_rolling_max_4w,Target_purchase_next_1w_rolling_mean_8w,Target_purchase_next_1w_rolling_std_8w,Target_purchase_next_1w_rolling_min_8w,Target_purchase_next_1w_rolling_max_8w,Target_purchase_next_1w_expanding_mean_1w,Target_purchase_next_1w_expanding_std_1w,Target_purchase_next_1w_expanding_min_1w,Target_purchase_next_1w_expanding_max_1w,Target_qty_next_1w_rolling_mean_4w,Target_qty_next_1w_rolling_std_4w,Target_qty_next_1w_rolling_min_4w,Target_qty_next_1w_rolling_max_4w,Target_qty_next_1w_rolling_mean_8w,Target_qty_next_1w_rolling_std_8w,Target_qty_next_1w_rolling_min_8w,Target_qty_next_1w_rolling_max_8w,Target_qty_next_1w_expanding_mean_1w,Target_qty_next_1w_expanding_std_1w,Target_qty_next_1w_expanding_min_1w,Target_qty_next_1w_expanding_max_1w,Target_purchase_next_2w_rolling_mean_4w_shift2,Target_purchase_next_2w_rolling_std_4w_shift2,Target_purchase_next_2w_rolling_min_4w_shift2,Target_purchase_next_2w_rolling_max_4w_shift2,Target_purchase_next_2w_rolling_mean_8w_shift2,Target_purchase_next_2w_rolling_std_8w_shift2,Target_purchase_next_2w_rolling_min_8w_shift2,Target_purchase_next_2w_rolling_max_8w_shift2,Target_purchase_next_2w_expanding_mean_2w,Target_purchase_next_2w_expanding_std_2w,Target_

,ID,customer_id,product_unit_variant_id,week_start,qty_this_week,num_orders_week,spend_this_week,purchased_this_week,product_id,grade_name,unit_name,product_grade_variant_id,selling_price,customer_category,customer_status,customer_created_at,Target_qty_next_1w,Target_purchase_next_1w,Target_qty_next_2w,Target_purchase_next_2w,product_name,product_grade_variant_sku,grade_active_status,is_train,qty_this_week_lag_2w,qty_this_week_lag_3w,qty_this_week_lag_4w,qty_this_week_lag_5w,qty_this_week_lag_6w,qty_this_week_lag_7w,qty_this_week_lag_8w,num_orders_week_lag_2w,num_orders_week_lag_3w,num_orders_week_lag_4w,num_orders_week_lag_5w,num_orders_week_lag_6w,num_orders_week_lag_7w,num_orders_week_lag_8w,spend_this_week_lag_2w,spend_this_week_lag_3w,spend_this_week_lag_4w,spend_this_week_lag_5w,spend_this_week_lag_6w,spend_this_week_lag_7w,spend_this_week_lag_8w,purchased_this_week_lag_2w,purchased_this_week_lag_3w,purchased_this_week_lag_4w,purchased_this_week_lag_5w,purchased_this_week_lag_6w,purchased_this_week_lag_7w,purchased_this_week_lag_8w,Target_purchase_next_2w_lag_2w,Target_purchase_next_2w_lag_3w,Target_purchase_next_2w_lag_4w,Target_purchase_next_2w_lag_5w,Target_purchase_next_2w_lag_6w,Target_purchase_next_2w_lag_7w,Target_purchase_next_2w_lag_8w,Target_qty_next_2w_lag_2w,Target_qty_next_2w_lag_3w,Target_qty_next_2w_lag_4w,Target_qty_next_2w_lag_5w,Target_qty_next_2w_lag_6w,Target_qty_next_2w_lag_7w,Target_qty_next_2w_lag_8w,weeks_since_last_purchase,last_purchase_qty,last_purchase_spend,qty_this_week_rolling_mean_4w,qty_this_week_rolling_std_4w,qty_this_week_rolling_min_4w,qty_this_week_rolling_max_4w,qty_this_week_rolling_mean_8w,qty_this_week_rolling_std_8w,qty_this_week_rolling_min_8w,qty_this_week_rolling_max_8w,qty_this_week_expanding_mean_1w,qty_this_week_expanding_std_1w,qty_this_week_expanding_min_1w,qty_this_week_expanding_max_1w,num_orders_week_rolling_mean_4w,num_orders_week_rolling_std_4w,num_orders_week_rolling_min_4w,num_orders_week_rolling_max_4w,num_orders_week_rolling_mean_8w,num_orders_week_rolling_std_8w,num_orders_week_rolling_min_8w,num_orders_week_rolling_max_8w,num_orders_week_expanding_mean_1w,num_orders_week_expanding_std_1w,num_orders_week_expanding_min_1w,num_orders_week_expanding_max_1w,spend_this_week_rolling_mean_4w,spend_this_week_rolling_std_4w,spend_this_week_rolling_min_4w,spend_this_week_rolling_max_4w,spend_this_week_rolling_mean_8w,spend_this_week_rolling_std_8w,spend_this_week_rolling_min_8w,spend_this_week_rolling_max_8w,spend_this_week_expanding_mean_1w,spend_this_week_expanding_std_1w,spend_this_week_expanding_min_1w,spend_this_week_expanding_max_1w,Target_purchase_next_1w_rolling_mean_4w,Target_purchase_next_1w_rolling_std_4w,Target_purchase_next_1w_rolling_min_4w,Target_purchase_next_1w_rolling_max_4w,Target_purchase_next_1w_rolling_mean_8w,Target_purchase_next_1w_rolling_std_8w,Target_purchase_next_1w_rolling_min_8w,Target_purchase_next_1w_rolling_max_8w,Target_purchase_next_1w_expanding_mean_1w,Target_purchase_next_1w_expanding_std_1w,Target_purchase_next_1w_expanding_min_1w,Target_purchase_next_1w_expanding_max_1w,Target_qty_next_1w_rolling_mean_4w,Target_qty_next_1w_rolling_std_4w,Target_qty_next_1w_rolling_min_4w,Target_qty_next_1w_rolling_max_4w,Target_qty_next_1w_rolling_mean_8w,Target_qty_next_1w_rolling_std_8w,Target_qty_next_1w_rolling_min_8w,Target_qty_next_1w_rolling_max_8w,Target_qty_next_1w_expanding_mean_1w,Target_qty_next_1w_expanding_std_1w,Target_qty_next_1w_expanding_min_1w,Target_qty_next_1w_expanding_max_1w,Target_purchase_next_2w_rolling_mean_4w_shift2,Target_purchase_next_2w_rolling_std_4w_shift2,Target_purchase_next_2w_rolling_min_4w_shift2,Target_purchase_next_2w_rolling_max_4w_shift2,Target_purchase_next_2w_rolling_mean_8w_shift2,Target_purchase_next_2w_rolling_std_8w_shift2,Target_purchase_next_2w_rolling_min_8w_shift2,Target_purchase_next_2w_rolling_max_8w_shift2,Target_purchase_next_2w_expanding_mean_2w,Target_purchase_next_2w_expanding_std_2w,Target_

In [ ]:
print("Training and Predicting with Horizon Method...")
preds = train_and_predict_horizon_stacked(train_df, test_df)



Training and Predicting with Horizon Method...
Initializing Gap-Aware Stacked Training...

Starting 3-Fold Validation (Mimicking 1-week gap)...


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU



--- Fold 1 Results ---
 Horizon 1 -> LGB(LogLoss 0.0576, AUC 0.9798) | CAT(LogLoss 0.0717, AUC 0.9815) | ENS(LogLoss 0.0625, AUC 0.9814) || MAE LGB 3.0074 | MAE CAT 3.1325 | MAE ENS 3.0579
 Horizon 2 -> LGB(LogLoss 0.0726, AUC 0.9686) | CAT(LogLoss 0.0920, AUC 0.9720) | ENS(LogLoss 0.0790, AUC 0.9721) || MAE LGB 3.3546 | MAE CAT 3.6078 | MAE ENS 3.4464


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU



--- Fold 2 Results ---
 Horizon 1 -> LGB(LogLoss 0.0589, AUC 0.9755) | CAT(LogLoss 0.0713, AUC 0.9774) | ENS(LogLoss 0.0628, AUC 0.9771) || MAE LGB 0.7590 | MAE CAT 0.8071 | MAE ENS 0.7499
 Horizon 2 -> LGB(LogLoss 0.0703, AUC 0.9709) | CAT(LogLoss 0.0889, AUC 0.9612) | ENS(LogLoss 0.0760, AUC 0.9683) || MAE LGB 3.3248 | MAE CAT 3.5565 | MAE ENS 3.3998


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU



--- Fold 3 Results ---
 Horizon 1 -> LGB(LogLoss 0.0588, AUC 0.9682) | CAT(LogLoss 0.0724, AUC 0.9692) | ENS(LogLoss 0.0633, AUC 0.9690) || MAE LGB 2.6956 | MAE CAT 2.8129 | MAE ENS 2.7161
 Horizon 2 -> LGB(LogLoss 0.0686, AUC 0.9743) | CAT(LogLoss 0.0825, AUC 0.9755) | ENS(LogLoss 0.0729, AUC 0.9758) || MAE LGB 3.7262 | MAE CAT 3.9864 | MAE ENS 3.7956

AVERAGE PERFORMANCE ACROSS ALL FOLDS
         logloss_lgb   auc_lgb  logloss_cat   auc_cat  logloss_ens   auc_ens  \
horizon                                                                        
1           0.058430  0.974483     0.071811  0.976022     0.062873  0.975811   
2           0.070507  0.971268     0.087778  0.969580     0.075962  0.972059   

          mae_lgb   mae_cat   mae_ens  
horizon                                
1        2.154016  2.250839  2.174658  
2        3.468537  3.716915  3.547296  
Best iters -> LGB CLF: 230, LGB REG: 177CAT CLF: 550, CAT REG: 1327


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


Generating Test Predictions (T+2 and T+3)...
Success.


In [ ]:
print("Creating submission...")
sub = pd.DataFrame({'ID': test_df['ID']})
for col in preds:
    sub[col] = preds[col]

sub.to_csv('hr31.csv', index=False)
print("Done! Submission saved to hr31.csv")

Creating submission...
Done! Submission saved to hr31.csv


In [ ]:
from google.colab import files
files.download('hr31.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
sub.head()

,ID,Target_purchase_next_1w_lgb,Target_purchase_next_1w_cat,Target_purchase_next_1w_ens,Target_purchase_next_2w_lgb,Target_purchase_next_2w_cat,Target_purchase_next_2w_ens,Target_qty_next_1w_lgb,Target_qty_next_1w_cat,Target_qty_next_1w_ens,Target_qty_next_2w_lgb,Target_qty_next_2w_cat,Target_qty_next_2w_ens
46,339_1_20250922,0.003681,0.004747,0.004214,0.003681,0.008929,0.006305,0.0,0.078807,0.039404,0.0,0.178179,0.089089
47,339_1_20250929,0.003681,0.004747,0.004214,0.003681,0.008929,0.006305,0.0,0.078807,0.039404,0.0,0.178179,0.089089
48,339_1_20251006,0.003681,0.005011,0.004346,0.003681,0.009424,0.006552,0.0,0.078747,0.039373,0.0,0.178071,0.089036
49,339_1_20251013,0.003681,0.005011,0.004346,0.003681,0.009424,0.006552,0.0,0.078747,0.039373,0.0,0.178071,0.089036
50,339_1_20251020,0.003681,0.005478,0.004579,0.003681,0.010187,0.006934,0.0,0.078747,0.039373,0.0,0.178071,0.089036


In [ ]:
 final_sub = sub[["ID", "Target_purchase_next_1w_ens", "Target_qty_next_1w_ens", "Target_purchase_next_2w_ens", "Target_qty_next_2w_ens" ]]
 final_sub.columns = ["ID", "Target_purchase_next_1w", "Target_qty_next_1w", "Target_purchase_next_2w", "Target_qty_next_2w"]
 final_sub.to_csv('ens_balanced.csv', index=False)
 final_sub.head()


,ID,Target_purchase_next_1w,Target_qty_next_1w,Target_purchase_next_2w,Target_qty_next_2w
46,339_1_20250922,0.004214,0.039404,0.006305,0.089089
47,339_1_20250929,0.004214,0.039404,0.006305,0.089089
48,339_1_20251006,0.004346,0.039373,0.006552,0.089036
49,339_1_20251013,0.004346,0.039373,0.006552,0.089036
50,339_1_20251020,0.004579,0.039373,0.006934,0.089036


In [ ]:
 final_sub = sub[["ID", "Target_purchase_next_1w_cat", "Target_qty_next_1w_cat", "Target_purchase_next_2w_cat", "Target_qty_next_2w_cat" ]]
 final_sub.columns = ["ID", "Target_purchase_next_1w", "Target_qty_next_1w", "Target_purchase_next_2w", "Target_qty_next_2w"]
 final_sub.to_csv('cat_balanced.csv', index=False)
 final_sub.head()


,ID,Target_purchase_next_1w,Target_qty_next_1w,Target_purchase_next_2w,Target_qty_next_2w
46,339_1_20250922,0.004747,0.078807,0.008929,0.178179
47,339_1_20250929,0.004747,0.078807,0.008929,0.178179
48,339_1_20251006,0.005011,0.078747,0.009424,0.178071
49,339_1_20251013,0.005011,0.078747,0.009424,0.178071
50,339_1_20251020,0.005478,0.078747,0.010187,0.178071


In [ ]:
 final_sub = sub[["ID", "Target_purchase_next_1w_lgb", "Target_qty_next_1w_lgb", "Target_purchase_next_2w_lgb", "Target_qty_next_2w_lgb" ]]
 final_sub.columns = ["ID", "Target_purchase_next_1w", "Target_qty_next_1w", "Target_purchase_next_2w", "Target_qty_next_2w"]
 final_sub.to_csv('lgb_balanced.csv', index=False)
 final_sub.head()


,ID,Target_purchase_next_1w,Target_qty_next_1w,Target_purchase_next_2w,Target_qty_next_2w
46,339_1_20250922,0.003681,0.0,0.003681,0.0
47,339_1_20250929,0.003681,0.0,0.003681,0.0
48,339_1_20251006,0.003681,0.0,0.003681,0.0
49,339_1_20251013,0.003681,0.0,0.003681,0.0
50,339_1_20251020,0.003681,0.0,0.003681,0.0


In [ ]:
 final_sub = sub[["ID", "Target_purchase_next_1w_cat", "Target_qty_next_1w_lgb", "Target_purchase_next_2w_cat", "Target_qty_next_2w_lgb" ]]
 final_sub.columns = ["ID", "Target_purchase_next_1w", "Target_qty_next_1w", "Target_purchase_next_2w", "Target_qty_next_2w"]
 final_sub.to_csv('cat_classification_lgb_regression_balanced.csv', index=False)
 final_sub.head()


,ID,Target_purchase_next_1w,Target_qty_next_1w,Target_purchase_next_2w,Target_qty_next_2w
46,339_1_20250922,0.004747,0.0,0.008929,0.0
47,339_1_20250929,0.004747,0.0,0.008929,0.0
48,339_1_20251006,0.005011,0.0,0.009424,0.0
49,339_1_20251013,0.005011,0.0,0.009424,0.0
50,339_1_20251020,0.005478,0.0,0.010187,0.0
